# RuleTaker and CLUTRR Dataset Collection for Neuro-Symbolic Reasoning Evaluation

## Description

This notebook demonstrates the collection, validation, and standardization of two datasets for neuro-symbolic reasoning pipeline evaluation:

1. **RuleTaker** (tasksource/ruletaker): Examples of logical reasoning over natural language rules. Each example contains a context (facts and rules) and a question to be evaluated as entailment or not entailment.

2. **CLUTRR** (tasksource/clutrr): Examples of relational reasoning over family relationships. Each example contains a story about family relationships and a query to predict the relationship between two entities.

The datasets are standardized to the `exp_sel_data_out.json` schema with `input`/`output` fields.

**Original Artifact**: Dataset collection script from the neuro-symbolic reasoning evaluation pipeline.

In [ ]:
# Install dependencies - only what's needed for this demo
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Only loguru is required by the original data.py script
_pip('loguru==0.7.2')

# Visualization packages (not in original script, added for demo)
_pip('matplotlib==3.10.0')

# Core packages pre-installed on Colab - only install locally
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2')

In [ ]:
# Original imports from data.py (preserved as-is)
from pathlib import Path
import json
import sys
import glob
import os

# Add logging
from loguru import logger
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

# Additional imports for notebook functionality
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

In [ ]:
# Data loading helper - uses GitHub URL with local fallback pattern
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-a4987d-uncertainty-aware-predicate-grounding-vi/main/round-1/dataset-1/demo/mini_demo_data.json"

def load_data():
    """Load mini_demo_data.json from GitHub URL with local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
# Load the demo data
data = load_data()
print(f"Loaded data with {len(data['datasets'])} datasets")
for dataset in data['datasets']:
    print(f"  - {dataset['dataset']}: {len(dataset['examples'])} examples")

## Configuration

Define tunable parameters for the dataset processing. Since this is a dataset collection script, the main parameters are related to which datasets to load and how many examples to include.

In [ ]:
# Configuration parameters - set to minimum for demo

# Which datasets to load (both by default)
LOAD_RULETAKER = True
LOAD_CLUTRR = True

# Maximum examples to load per dataset (None = all)
# Set to small number for demo purposes
MAX_EXAMPLES_PER_DATASET = None  # Set to e.g. 10 for testing

# Output file name
OUTPUT_FILE = "demo_output.json"

print("Configuration set:")
print(f"  Load RuleTaker: {LOAD_RULETAKER}")
print(f"  Load CLUTRR: {LOAD_CLUTRR}")
print(f"  Max examples per dataset: {MAX_EXAMPLES_PER_DATASET}")

## Process RuleTaker Dataset

The RuleTaker dataset contains examples of logical reasoning over natural language rules. Each example has:
- **input**: A context with facts and rules, plus a statement to evaluate
- **output**: "entailment" or "not entailment"

This section processes the RuleTaker examples from the loaded data.

In [ ]:
# Process RuleTaker dataset (adapted from data.py)

result = {"datasets": []}

if LOAD_RULETAKER:
    logger.info("Processing RuleTaker dataset...")
    
    # Find RuleTaker in loaded data
    ruletaker_data = None
    for dataset in data['datasets']:
        if dataset['dataset'] == 'ruletaker':
            ruletaker_data = dataset
            break
    
    if ruletaker_data:
        ruletaker_examples = ruletaker_data['examples']
        
        # Limit examples if configured
        if MAX_EXAMPLES_PER_DATASET is not None:
            ruletaker_examples = ruletaker_examples[:MAX_EXAMPLES_PER_DATASET]
        
        result["datasets"].append({
            "dataset": "ruletaker",
            "examples": ruletaker_examples
        })
        logger.info(f"Loaded {len(ruletaker_examples)} examples from RuleTaker")
    else:
        logger.warning("RuleTaker dataset not found in loaded data")

print(f"Result now has {len(result['datasets'])} datasets")

## Process CLUTRR Dataset

The CLUTRR dataset contains examples of relational reasoning over family relationships. Each example has:
- **input**: A story about family relationships
- **output**: A relationship code (integer as string)

This section processes the CLUTRR examples from the loaded data.

In [ ]:
# Process CLUTRR dataset (adapted from data.py)

if LOAD_CLUTRR:
    logger.info("Processing CLUTRR dataset...")
    
    # Find CLUTRR in loaded data
    clutrr_data = None
    for dataset in data['datasets']:
        if dataset['dataset'] == 'clutrr':
            clutrr_data = dataset
            break
    
    if clutrr_data:
        clutrr_examples = clutrr_data['examples']
        
        # Limit examples if configured
        if MAX_EXAMPLES_PER_DATASET is not None:
            clutrr_examples = clutrr_examples[:MAX_EXAMPLES_PER_DATASET]
        
        result["datasets"].append({
            "dataset": "clutrr",
            "examples": clutrr_examples
        })
        logger.info(f"Loaded {len(clutrr_examples)} examples from CLUTRR")
    else:
        logger.warning("CLUTRR dataset not found in loaded data")

print(f"Result now has {len(result['datasets'])} datasets")

## Save Output

Save the processed datasets to a combined output file (mirroring the original script's behavior).

In [ ]:
# Save output (combined) - adapted from data.py

output_file = Path(OUTPUT_FILE)
with open(output_file, 'w') as f:
    json.dump(result, f, indent=2)

logger.info(f"Saved {len(result['datasets'])} datasets to {output_file}")
total_examples = sum(len(d["examples"]) for d in result["datasets"])
logger.info(f"Total examples: {total_examples}")

## Visualization and Summary

Display key statistics and sample data from the processed datasets.

In [ ]:
# Visualization and summary of the datasets

print("="*60)
print("DATASET SUMMARY")
print("="*60)

for dataset in result['datasets']:
    dataset_name = dataset['dataset']
    examples = dataset['examples']
    
    print(f"\nDataset: {dataset_name}")
    print(f"  Number of examples: {len(examples)}")
    
    # Analyze outputs
    outputs = [ex['output'] for ex in examples]
    output_counts = Counter(outputs)
    
    print(f"  Output distribution:")
    for output, count in output_counts.most_common():
        print(f"    {output}: {count}")
    
    # Show sample examples
    print(f"\n  Sample examples:")
    for i, example in enumerate(examples[:2]):  # Show first 2 examples
        print(f"\n    Example {i+1}:")
        print(f"      Input: {example['input'][:100]}..." if len(example['input']) > 100 else f"      Input: {example['input']}")
        print(f"      Output: {example['output']}")

print("\n" + "="*60)
print("END OF SUMMARY")
print("="*60)

In [ ]:
# Visualize output distribution for each dataset

fig, axes = plt.subplots(1, len(result['datasets']), figsize=(5*len(result['datasets']), 4))
if len(result['datasets']) == 1:
    axes = [axes]

for idx, dataset in enumerate(result['datasets']):
    examples = dataset['examples']
    outputs = [ex['output'] for ex in examples]
    output_counts = Counter(outputs)
    
    # Prepare data for plotting
    labels = list(output_counts.keys())
    counts = list(output_counts.values())
    
    ax = axes[idx]
    bars = ax.bar(range(len(labels)), counts, color=['blue', 'orange', 'green'][:len(labels)])
    ax.set_xlabel('Output Class')
    ax.set_ylabel('Count')
    ax.set_title(f'{dataset["dataset"].upper()} Output Distribution')
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha='right')
    
    # Add count labels on bars
    for bar, count in zip(bars, counts):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.1,
                f'{count}', ha='center', va='bottom')

plt.tight_layout()
plt.show()